# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from tqdm.notebook import tqdm
from tqdm import tqdm
from sklearn.metrics import accuracy_score

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv', sep=',')
dow = pd.read_csv('../data/dayofweek.csv', sep=',')

X = df
y = dow['dayofweek']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [4]:
clf = SVC(probability=True, random_state=21)

In [5]:
param_grid = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C' : [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

In [6]:
grid = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=2,                  # 2-fold cross-validation
    scoring='accuracy',
    n_jobs=-1               # использовать все ядра CPU
)

In [7]:
grid.fit(X_train, y_train)

,estimator,SVC(probabili...ndom_state=21)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': ['balanced', None], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf', ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,2
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


In [8]:
print(f'Best parameters: {grid.best_params_}')
print(f'Best accuracy: {grid.best_score_:.5f}')

Best parameters: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best accuracy: 0.80935


In [9]:
results = pd.DataFrame(grid.cv_results_)
results_sorted = results.sort_values(by='rank_test_score')

In [10]:
results_sorted[['param_kernel', 'param_C', 'param_gamma', 'param_class_weight',
                'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)

,param_kernel,param_C,param_gamma,param_class_weight,mean_test_score,std_test_score,rank_test_score
70,rbf,10.0,auto,None,0.809347,0.003709,1
64,rbf,10.0,auto,balanced,0.804154,0.002967,2
58,rbf,5.0,auto,None,0.747774,0.002967,3
52,rbf,5.0,auto,balanced,0.737389,0.004451,4
60,linear,10.0,scale,balanced,0.706973,0.005193,5
63,linear,10.0,auto,balanced,0.706973,0.005193,5
66,linear,10.0,scale,None,0.695104,0.011128,7
69,linear,10.0,auto,None,0.695104,0.011128,7
51,linear,5.0,auto,balanced,0.686944,0.019288,9
48,linear,5.0,scale,balanced,0.686944,0.019288,9


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [27]:
dt = DecisionTreeClassifier(random_state=21)

In [28]:
param_grid = {
    'max_depth': list(range(1,50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

In [29]:
grid = GridSearchCV(
    estimator = dt,
    param_grid=param_grid,
    scoring = 'accuracy',
    cv=10,
    n_jobs=-1
)

In [30]:
grid.fit(X_train, y_train)

,estimator,DecisionTreeC...ndom_state=21)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'], 'max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [31]:
print(f'Best parameters: {grid.best_params_}')
print(f'Best accuracy: {grid.best_score_:.5f}')

Best parameters: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 25}
Best accuracy: 0.89018


In [32]:
results = pd.DataFrame(grid.cv_results_)
results_sorted = results.sort_values(by='rank_test_score')

In [33]:
results_sorted[['param_max_depth', 'param_class_weight', 'param_criterion',
                'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)

,param_max_depth,param_class_weight,param_criterion,mean_test_score,std_test_score,rank_test_score
24,25,balanced,entropy,0.890182,0.017012,1
25,26,balanced,entropy,0.890182,0.017012,1
26,27,balanced,entropy,0.890182,0.017012,1
27,28,balanced,entropy,0.890182,0.017012,1
28,29,balanced,entropy,0.890182,0.017012,1
29,30,balanced,entropy,0.890182,0.017012,1
30,31,balanced,entropy,0.890182,0.017012,1
31,32,balanced,entropy,0.890182,0.017012,1
39,40,balanced,entropy,0.890182,0.017012,1
38,39,balanced,entropy,0.890182,0.017012,1


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [34]:
rf = RandomForestClassifier(random_state=21)

param_grid = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

In [35]:
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=10,
    n_jobs=-1
)

grid.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=21)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'], 'max_depth': [1, 2, ...], 'n_estimators': [5, 10, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [20]:
print(f'Best parameters: {grid.best_params_}')
print(f'Best accuracy: {grid.best_score_:.5f}')

Best parameters: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 26, 'n_estimators': 100}
Best accuracy: 0.87685


In [21]:
results = pd.DataFrame(grid.cv_results_)

results_sorted = results.sort_values(by='rank_test_score')

results_sorted[['param_n_estimators', 'param_max_depth',
                'param_class_weight', 'param_criterion',
                'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)

,param_n_estimators,param_max_depth,param_class_weight,param_criterion,mean_test_score,std_test_score,rank_test_score
103,100,26,balanced,entropy,0.876855,0.008902,1
523,100,33,None,entropy,0.874629,0.009644,2
507,100,29,None,entropy,0.874629,0.009644,2
503,100,28,None,entropy,0.874629,0.009644,2
511,100,30,None,entropy,0.873887,0.008902,5
111,100,28,balanced,entropy,0.873887,0.007418,5
499,100,27,None,entropy,0.873887,0.008902,5
115,100,29,balanced,entropy,0.873145,0.008160,8
487,100,24,None,entropy,0.873145,0.008160,8
515,100,31,None,entropy,0.873145,0.011128,8


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [22]:
n_estimators_list = [5, 10, 50, 100]
max_depth_list = list(range(1, 50))
class_weight_list = ['balanced', None]
criterion_list = ['entropy', 'gini']

In [23]:
results = []

for n_est in tqdm(n_estimators_list, desc="n_estimators"):
    for depth in max_depth_list:
        for weight in class_weight_list:
            for crit in criterion_list:
                model = RandomForestClassifier(
                    n_estimators=n_est,
                    max_depth=depth,
                    class_weight=weight,
                    criterion=crit,
                    random_state=21,
                    n_jobs=-1
                )

                # Кросс-валидация
                scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
                mean_acc = np.mean(scores)
                std_acc = np.std(scores)

                results.append({
                    'n_estimators': n_est,
                    'max_depth': depth,
                    'class_weight': weight,
                    'criterion': crit,
                    'mean_accuracy': mean_acc,
                    'std_accuracy': std_acc
                })

n_estimators: 100%|██████████| 4/4 [09:25<00:00, 141.26s/it]


In [24]:
results_df = pd.DataFrame(results)

results_sorted = results_df.sort_values(by='mean_accuracy', ascending=False)

results_sorted.head(10)

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
503,50,28,None,gini,0.904290,0.010961
711,100,31,None,gini,0.903547,0.014380
509,50,30,balanced,gini,0.902817,0.013554
525,50,34,balanced,gini,0.902809,0.013010
743,100,39,None,gini,0.902806,0.010460
751,100,41,None,gini,0.902806,0.010460
731,100,36,None,gini,0.902806,0.010460
735,100,37,None,gini,0.902806,0.010460
759,100,43,None,gini,0.902806,0.010460
775,100,47,None,gini,0.902806,0.010460


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [25]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

In [26]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Final test accuracy: {accuracy:.4f}")

Final test accuracy: 0.9320
